In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
#takes ~5 min
!apt-get install -y aria2 p7zip-full

!aria2c -x 16 -s 16 -k 1M -q -o msmd.zip.part \
"https://zenodo.org/record/2597505/files/msmd_aug_v1-1_no-audio.zip"

!7z x msmd.zip.part -o/content/msmd_dataset -mmt=on

In [ ]:
!pip install muscima
!pip install torch-geometric

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, global_mean_pool
from torchvision.models import swin_v2_t, Swin_V2_T_Weights

import os
import glob
import numpy as np
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T
import muscima
import muscima.io
import copy
import random

import torch.optim as optim
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import defaultdict
from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
import torchaudio.transforms as T_audio
from sklearn.preprocessing import LabelEncoder
import time

In [ ]:
class_vocab = {
    '<UNK>': 0,
    '16th_flag': 1,
    '16th_rest': 2,
    '8th_flag': 3,
    '8th_rest': 4,
    'accent': 5,
    'beam': 6,
    'c-clef': 7,
    'double_sharp': 8,
    'duration-dot': 9,
    'dynamics_text': 10,
    'f-clef': 11,
    'flat': 12,
    'g-clef': 13,
    'grace_strikethrough': 14,
    'grace-notehead-full': 15,
    'hairpin-cresc.': 16,
    'hairpin-decr.': 17,
    'half_rest': 18,
    'key_signature': 19,
    'ledger_line': 20,
    'letter_a': 21,
    'letter_c': 22,
    'letter_d': 23,
    'letter_e': 24,
    'letter_f': 25,
    'letter_i': 26,
    'letter_l': 27,
    'letter_M': 28,
    'letter_m': 29,
    'letter_n': 30,
    'letter_o': 31,
    'letter_P': 32,
    'letter_p': 33,
    'letter_r': 34,
    'letter_s': 35,
    'letter_t': 36,
    'letter_u': 37,
    'measure_separator': 38,
    'multi-staff_brace': 39,
    'multi-staff_bracket': 40,
    'multiple-note_tremolo': 41,
    'natural': 42,
    'notehead-empty': 43,
    'notehead-full': 44,
    'numeral_2': 45,
    'numeral_3': 46,
    'numeral_4': 47,
    'numeral_5': 48,
    'numeral_6': 49,
    'numeral_7': 50,
    'numeral_8': 51,
    'ornament(s)': 52,
    'other_text': 53,
    'other-dot': 54,
    'quarter_rest': 55,
    'repeat': 56,
    'repeat-dot': 57,
    'sharp': 58,
    'slur': 59,
    'staccato-dot': 60,
    'staff_grouping': 61,
    'stem': 62,
    'tempo_text': 63,
    'tenuto': 64,
    'thin_barline': 65,
    'tie': 66,
    'time_signature': 67,
    'trill': 68,
    'tuple': 69,
    'tuple_bracket/line': 70,
    'whole_rest': 71
}

In [ ]:
def get_deterministic_splits(root_dir, train_ratio=0.75, val_ratio=0.10):
    """
    Sorts piece directories alphabetically and assigns them to splits
    """
    pieces = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

    rng = random.Random(42)
    rng.shuffle(pieces)

    n = len(pieces)
    train_end = int(train_ratio * n)
    val_end = train_end + int(val_ratio * n)

    train_pieces = pieces[:train_end]
    val_pieces = pieces[train_end:val_end]
    test_pieces = pieces[val_end:]

    return train_pieces, val_pieces, test_pieces

### Preprocess and save + zip images

In [ ]:
DATASET_ROOT = '/content/msmd_dataset/msmd_aug_v1-1_no-audio'

train_pieces, val_pieces, test_pieces = get_deterministic_splits(DATASET_ROOT)
def preprocess_and_save_images(
    root_dir,
    split_pieces,
    save_dir='/content/drive/MyDrive/MSMD_Checkpoints/data'
):
    os.makedirs(save_dir, exist_ok=True)
    print(f"Saving preprocessed images to: {save_dir}")

    TARGET_W = 416
    TARGET_H = 128

    processed_count = 0

    print(f"Processing {len(split_pieces)} pieces...")

    for piece_name in tqdm(split_pieces, desc="Extracting Staff Lines"):
        piece_dir = os.path.join(root_dir, piece_name)
        if not os.path.isdir(piece_dir): continue

        score_dir_base = os.path.join(piece_dir, 'scores')
        score_folders = os.listdir(score_dir_base)
        if not score_folders: continue
        score_path = os.path.join(score_dir_base, score_folders[0])

        img_dir = os.path.join(score_path, 'img')
        coords_dir = os.path.join(score_path, 'coords')

        if not os.path.exists(img_dir) or not os.path.exists(coords_dir): continue

        for img_file in glob.glob(os.path.join(img_dir, '*.png')):
            page_id = os.path.basename(img_file).replace('.png', '')
            systems_npy_path = os.path.join(coords_dir, f"systems_{page_id}.npy")

            if not os.path.exists(systems_npy_path): continue

            system_boxes = np.load(systems_npy_path)

            with Image.open(img_file) as img:
                # Convert to Grayscale immediately to save RAM and Drive space
                img = img.convert('L')
                img_width, img_height = img.size

                for box in system_boxes:
                    if np.ptp(box[:, 0]) < np.ptp(box[:, 1]):
                        sys_top, sys_bottom = float(np.min(box[:, 0])), float(np.max(box[:, 0]))
                    else:
                        sys_top, sys_bottom = float(np.min(box[:, 1])), float(np.max(box[:, 1]))

                    crop_top = max(0, int(sys_top - 30))
                    crop_bottom = min(img_height, int(sys_bottom + 30))

                    crop_img = img.crop((0, crop_top, img_width, crop_bottom))
                    orig_w, orig_h = crop_img.size

                    scale = min(TARGET_W / orig_w, TARGET_H / orig_h)
                    new_w = int(orig_w * scale)
                    new_h = int(orig_h * scale)

                    resized_img = crop_img.resize((new_w, new_h), Image.Resampling.LANCZOS)

                    canvas = Image.new('L', (TARGET_W, TARGET_H), color=255)

                    paste_x = (TARGET_W - new_w) // 2
                    paste_y = (TARGET_H - new_h) // 2
                    canvas.paste(resized_img, (paste_x, paste_y))

                    line_id = f"{piece_name}_{page_id}_{sys_top}"

                    save_path = os.path.join(save_dir, f"{line_id}.png")

                    canvas.save(save_path, format="PNG", optimize=True)
                    processed_count += 1

    print(f"\nSuccessfully generated and saved {processed_count} to Drive!")

preprocess_and_save_images(root_dir='/content/msmd_dataset/msmd_aug_v1-1_no-audio', split_pieces=train_pieces)
preprocess_and_save_images(root_dir='/content/msmd_dataset/msmd_aug_v1-1_no-audio', split_pieces=val_pieces)
preprocess_and_save_images(root_dir='/content/msmd_dataset/msmd_aug_v1-1_no-audio', split_pieces=test_pieces)

In [ ]:
# !cd /content/drive/MyDrive/MSMD_Checkpoints/ && zip -r -q data.zip data/

### Locally load zipped images

In [ ]:
!mkdir -p /content/local_data

!cp /content/drive/MyDrive/MSMD_Checkpoints/data.zip /content/

!unzip -q /content/data.zip -d /content/local_data

!rm /content/data.zip

### Dataset + dataloaders

In [ ]:
class MSMDDataset(Dataset):
    def __init__(self, root_dir, split_pieces, class_vocab, mode='train', num_crops=2, min_frames=80, max_frames=160, val_frames=200, transform=None):
        self.root_dir = root_dir
        self.mode = mode
        self.num_crops = num_crops if mode == 'train' else 1
        self.min_frames = min_frames # 4 seconds
        self.max_frames = max_frames # 8 seconds
        self.val_frames = val_frames # 10 seconds

        self.transform = transform or T.Compose([T.ToTensor()])
        self.class_vocab = class_vocab
        self.data_index = []

        print(f"Indexing {len(split_pieces)} pieces")

        for piece_name in tqdm(split_pieces, desc="Pre-building Graph Tensors"):
            piece_dir = os.path.join(root_dir, piece_name)
            if not os.path.isdir(piece_dir): continue

            perf_dir = os.path.join(piece_dir, 'performances')
            performances = [p for p in os.listdir(perf_dir) if os.path.isdir(os.path.join(perf_dir, p))]

            score_dir_base = os.path.join(piece_dir, 'scores')
            score_folders = os.listdir(score_dir_base)
            if not score_folders: continue
            score_path = os.path.join(score_dir_base, score_folders[0])

            xml_files = sorted(glob.glob(os.path.join(score_path, 'mung', '*.xml')))

            for xml_file in xml_files:
                page_id = os.path.basename(xml_file).replace('.xml', '')
                systems_npy_path = os.path.join(score_path, 'coords', f"systems_{page_id}.npy")
                if not os.path.exists(systems_npy_path): continue

                img_path = os.path.join(score_path, 'img', f"{page_id}.png")
                with Image.open(img_path) as img:
                    image_width = float(img.size[0])

                system_boxes = np.load(systems_npy_path)
                nodes = muscima.io.parse_cropobject_list(xml_file)

                for perf_name in performances:
                    spec_path = os.path.join(perf_dir, perf_name, 'features', f"{perf_name}.flac_spec.npy")
                    notes_path = os.path.join(perf_dir, perf_name, 'features', f"{perf_name}.flac_notes.npy")

                    if not os.path.exists(notes_path) or not os.path.exists(spec_path):
                        continue

                    notes_array = np.load(notes_path)
                    onset_key = f"{perf_name}_onset_frame"
                    event_idx_key = f"{perf_name}_note_event_idx"

                    for box in system_boxes:
                        if np.ptp(box[:, 0]) < np.ptp(box[:, 1]):
                            sys_top, sys_bottom = float(np.min(box[:, 0])), float(np.max(box[:, 0]))
                        else:
                            sys_top, sys_bottom = float(np.min(box[:, 1])), float(np.max(box[:, 1]))

                        crop_top = max(0, sys_top - 30)
                        crop_bottom = sys_bottom + 30

                        line_id = f"{piece_name}_{page_id}_{sys_top}"

                        valid_nodes = [
                            n for n in nodes
                            if crop_top <= n.top <= crop_bottom
                            and n.clsname not in ['system', 'staff', 'measure', 'staff_space']
                        ]
                        if not valid_nodes: continue

                        min_start_frame = float('inf')
                        max_end_frame = 0
                        played_notes = []

                        for n in valid_nodes:
                            if n.clsname == 'notehead-full' and onset_key in n.data:
                                played_notes.append(n)
                                onset_frame = int(n.data[onset_key])
                                min_start_frame = min(min_start_frame, onset_frame)

                                event_idx = n.data.get(event_idx_key)
                                if event_idx is not None and event_idx < len(notes_array):
                                    duration_frames = int(float(notes_array[event_idx, 2]) * 20.0)
                                    max_end_frame = max(max_end_frame, onset_frame + duration_frames)
                                else:
                                    max_end_frame = max(max_end_frame, onset_frame + 20)

                        if min_start_frame == float('inf'):
                            continue

                        start_frame = int(min_start_frame)
                        end_frame = int(max_end_frame) + 5

                        record_info = {
                            'crop_top': crop_top,
                            'crop_bottom': crop_bottom,
                            'start_frame': start_frame,
                            'end_frame': end_frame,
                            'perf_name': perf_name
                        }

                        x_cont, x_class, x_pitch, edge_index = self._build_graph_from_nodes(
                            valid_nodes, played_notes, record_info, image_width, notes_array, onset_key, event_idx_key
                        )
                        img_save_dir = '/content/local_data'

                        self.data_index.append({
                            'spec_path': spec_path,
                            'image_path': os.path.join(img_save_dir, f"{line_id}.png"),
                            'start_frame': start_frame,
                            'end_frame': end_frame,
                            'graph_x_cont': x_cont,
                            'graph_x_class': x_class,
                            'graph_x_pitch': x_pitch,
                            'graph_edge_index': edge_index,
                            'line_id': line_id
                        })

        print(f"Successfully cached {len(self.data_index)} graph tensors in RAM.")

    def __len__(self):
        return len(self.data_index)

    def _build_graph_from_nodes(self, valid_nodes, played_notes, record, image_width, notes_array, onset_key, event_idx_key):
        id_to_idx = {node.objid: i for i, node in enumerate(valid_nodes)}
        crop_height = record['crop_bottom'] - record['crop_top']
        max_x = float(image_width)

        x_cont_list, x_class_list, x_pitch_list = [], [], []
        src_edges, dst_edges = [], []

        valid_onsets = [notes_array[n.data[event_idx_key], 0] for n in played_notes if event_idx_key in n.data]
        base_time_sec = min(valid_onsets) if valid_onsets else 0.0
        line_duration_sec = (record['end_frame'] - record['start_frame']) / 20.0

        for node in valid_nodes:
            top_in_crop = max(0, node.top - record['crop_top'])
            bottom_in_crop = min(crop_height, (node.top + node.height) - record['crop_top'])

            norm_top = top_in_crop / crop_height
            norm_height = max(0, bottom_in_crop - top_in_crop) / crop_height
            norm_left = node.left / max_x
            norm_width = node.width / max_x

            event_idx = node.data.get(event_idx_key)
            if event_idx is not None and event_idx < len(notes_array):
                onset_sec = float(notes_array[event_idx, 0])
                duration_sec = float(notes_array[event_idx, 2])
                relative_onset_sec = onset_sec - base_time_sec
                norm_onset = max(0.0, min(1.0, relative_onset_sec / line_duration_sec))
                norm_duration = max(0.0, min(1.0, duration_sec / line_duration_sec))
            else:
                norm_duration = 0.0
                norm_onset = 0.0

            x_cont_list.append([norm_top, norm_left, norm_width, norm_height, norm_duration, norm_onset])
            x_class_list.append(self.class_vocab.get(node.clsname, 0))
            x_pitch_list.append(int(node.data.get('midi_pitch_code', 0)))

            src_idx = id_to_idx[node.objid]
            for target_id in node.outlinks:
                if target_id in id_to_idx:
                    dst_idx = id_to_idx[target_id]
                    src_edges.extend([src_idx, dst_idx])
                    dst_edges.extend([dst_idx, src_idx])

        time_groups = defaultdict(list)
        for n in played_notes:
            time_groups[n.data[onset_key]].append(n)

        sorted_times = sorted(time_groups.keys())
        for i, current_time in enumerate(sorted_times):
            current_chord_notes = time_groups[current_time]

            if len(current_chord_notes) > 1:
                for idx_a in range(len(current_chord_notes)):
                    for idx_b in range(idx_a + 1, len(current_chord_notes)):
                        src_idx = id_to_idx[current_chord_notes[idx_a].objid]
                        dst_idx = id_to_idx[current_chord_notes[idx_b].objid]
                        src_edges.extend([src_idx, dst_idx])
                        dst_edges.extend([dst_idx, src_idx])

            if i < len(sorted_times) - 1:
                next_time = sorted_times[i + 1]
                next_chord_notes = time_groups[next_time]
                for curr_n in current_chord_notes:
                    for next_n in next_chord_notes:
                        src_idx = id_to_idx[curr_n.objid]
                        dst_idx = id_to_idx[next_n.objid]
                        src_edges.append(src_idx)
                        dst_edges.append(dst_idx)

        x_cont = torch.tensor(x_cont_list, dtype=torch.float32)
        x_class = torch.tensor(x_class_list, dtype=torch.long)
        x_pitch = torch.tensor(x_pitch_list, dtype=torch.long)
        edge_index = torch.tensor([src_edges, dst_edges], dtype=torch.long) if src_edges else torch.empty((2, 0), dtype=torch.long)

        return x_cont, x_class, x_pitch, edge_index

    def __getitem__(self, idx):
        record = self.data_index[idx]

        start_frame = record['start_frame']
        end_frame = record['end_frame']
        total_frames = end_frame - start_frame

        full_spec = np.load(record['spec_path'], mmap_mode='r')
        spec_slice = full_spec[:, start_frame : end_frame]
        spec_slice = (spec_slice - np.mean(spec_slice)) / (np.std(spec_slice) + 1e-6)
        spec_tensor = torch.tensor(np.copy(spec_slice), dtype=torch.float32).unsqueeze(0)

        crops = []

        if self.mode == 'val':
            if total_frames <= self.val_frames:
                crops.append(spec_tensor)
            else:
                center_point = total_frames // 2
                half_crop = self.val_frames // 2
                start_idx = center_point - half_crop
                end_idx = start_idx + self.val_frames
                crops.append(spec_tensor[:, :, start_idx:end_idx])

        else:
            if total_frames <= self.min_frames:
                for _ in range(self.num_crops):
                    crops.append(spec_tensor)
            else:
                for _ in range(self.num_crops):
                    current_crop_len = random.randint(self.min_frames, min(self.max_frames, total_frames))
                    max_start = total_frames - current_crop_len
                    random_start = random.randint(0, max_start)
                    random_end = random_start + current_crop_len
                    crops.append(spec_tensor[:, :, random_start:random_end])


        img = Image.open(record['image_path']).convert('RGB')
        img_tensor = self.transform(img)
        return {
            'image': img_tensor,
            'spectrogram_crops': crops,
            'graph_x_cont': record['graph_x_cont'],
            'graph_x_class': record['graph_x_class'],
            'graph_x_pitch': record['graph_x_pitch'],
            'graph_edge_index': record['graph_edge_index'],
            'line_id': record['line_id']
        }

In [ ]:
def custom_collate_fn(batch):
    max_time_frames = max([crop.shape[2] for item in batch for crop in item['spectrogram_crops']])

    padded_spectrograms = []
    images = []

    x_cont_list, x_class_list, x_pitch_list = [], [], []
    edge_index_list, batch_index_list = [], []

    node_offset = 0
    batch_index_counter = 0
    line_ids = []

    for item in batch:
        for spec_crop in item['spectrogram_crops']:
            images.append(item['image'])
            seq_len = spec_crop.shape[2]
            pad_amount = max_time_frames - seq_len
            padded_spec = F.pad(spec_crop, (0, pad_amount))
            padded_spectrograms.append(padded_spec)

            num_nodes = item['graph_x_cont'].shape[0]
            x_cont_list.append(item['graph_x_cont'])
            x_class_list.append(item['graph_x_class'])
            x_pitch_list.append(item['graph_x_pitch'])

            if item['graph_edge_index'].numel() > 0:
                edge_index_list.append(item['graph_edge_index'] + node_offset)

            batch_index_list.append(torch.full((num_nodes,), batch_index_counter, dtype=torch.long))

            node_offset += num_nodes
            batch_index_counter += 1
            line_ids.append(item['line_id'])

    return {
        'images': torch.stack(images),
        'spectrograms': torch.stack(padded_spectrograms), # [B * num_crops, 1, 92, max_frames]
        'graph_x_cont': torch.cat(x_cont_list, dim=0),
        'graph_x_class': torch.cat(x_class_list, dim=0),
        'graph_x_pitch': torch.cat(x_pitch_list, dim=0),
        'graph_edge_index': torch.cat(edge_index_list, dim=1) if edge_index_list else torch.empty((2,0), dtype=torch.long),
        'graph_batch_index': torch.cat(batch_index_list, dim=0),
        'line_id': line_ids
    }

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
DATASET_ROOT = '/content/msmd_dataset/msmd_aug_v1-1_no-audio'
set_seed()
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(42)

train_pieces, val_pieces, test_pieces = get_deterministic_splits(DATASET_ROOT)
print(f"Split sizes -> Train: {len(train_pieces)} | Val: {len(val_pieces)} | Test: {len(test_pieces)}")

train_dataset = MSMDDataset(
    root_dir=DATASET_ROOT,
    split_pieces=train_pieces,
    class_vocab=class_vocab
)

val_dataset = MSMDDataset(
    root_dir=DATASET_ROOT,
    split_pieces=val_pieces,
    class_vocab=class_vocab,
    mode = 'val'
)

# test_dataset = MSMDDataset(
#     root_dir=DATASET_ROOT,
#     split_pieces=test_pieces,
#     class_vocab=class_vocab,
#     mode = 'val'
# )

BATCH_SIZE = 32 #16#64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=custom_collate_fn,
    drop_last=True,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=g
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=custom_collate_fn,
    drop_last=False,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker
)

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     collate_fn=custom_collate_fn,
#     drop_last=False,
#     num_workers=2,
#     pin_memory=True,
#     worker_init_fn=seed_worker
# )

### Stage 1
Audio-Graph Pretraining

In [ ]:
class SheetMusicTeacherGAT(nn.Module):
    def __init__(self, num_classes, num_pitches=129, embed_dim=64, hidden_channels=128, out_channels=512, heads=4):
        super(SheetMusicTeacherGAT, self).__init__()


        self.class_embedding = nn.Embedding(num_embeddings=num_classes, embedding_dim=embed_dim)
        self.pitch_embedding = nn.Embedding(num_embeddings=num_pitches, embedding_dim=embed_dim)

        # 7 (continuous) + 64 (class) + 64 (pitch)
        in_channels = 6 + (embed_dim * 2)

        # output dim = hidden_channels * heads = 128 * 4 = 512
        self.gat1 = GATv2Conv(in_channels, hidden_channels, heads=heads, concat=True, dropout=0.1)
        self.norm1 = nn.LayerNorm(hidden_channels * heads)
        # output dim = 256
        self.gat2 = GATv2Conv(hidden_channels * heads, 256, heads=1, concat=False, dropout=0.1)
        self.norm2 = nn.LayerNorm(256)

        self.projection_head = nn.Sequential(
            nn.Linear(256, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, out_channels)
        )

    def forward(self, x_cont, x_class, x_pitch, edge_index, batch):
        """
        x_cont: Tensor [num_nodes, 5] (Spatial coordinates & duraiton)
        x_class: Tensor [num_nodes] (Integer IDs for symbol classes)
        x_pitch: Tensor [num_nodes] (Integer IDs for MIDI pitches)
        edge_index: Tensor [2, num_edges] (Connectivity from Outlinks)
        batch: Tensor [num_nodes] (Maps each node to its respective graph in the batch)
        """

        class_emb = self.class_embedding(x_class)  # [num_nodes, 64]
        pitch_emb = self.pitch_embedding(x_pitch)  # [num_nodes, 64]

        # 2. Concatenate all node features
        x = torch.cat([x_cont, class_emb, pitch_emb], dim=1)  # [num_nodes, 133]

        x = self.gat1(x, edge_index)
        x = self.norm1(x)
        x = F.elu(x)

        x = self.gat2(x, edge_index)
        x = self.norm2(x)
        x = F.elu(x)

        x_pooled = global_mean_pool(x, batch)  # [batch_size, 256]
        out = self.projection_head(x_pooled) # [batch_size, 512]
        out = F.normalize(out, p=2, dim=1)

        return out

In [ ]:
class SheetMusicSwin(nn.Module):
    def __init__(self, out_channels=512):
        super(SheetMusicSwin, self).__init__()

        self.backbone = swin_v2_t(weights=Swin_V2_T_Weights.DEFAULT)

        in_features = self.backbone.head.in_features

        self.backbone.head = nn.Identity()

        self.projection_head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, out_channels)
        )

    def forward(self, x):
        """
        x: Tensor of shape [batch_size, 3, H, W]
        """
        #[batch_size, 1024]
        features = self.backbone(x)

        # [batch_size, 512])
        projected = self.projection_head(features)

        out = F.normalize(projected, p=2, dim=1)

        return out

In [ ]:
class SpectrogramSwin(nn.Module):
    def __init__(self, out_channels=512):
        super(SpectrogramSwin, self).__init__()

        self.backbone = swin_v2_t(weights=Swin_V2_T_Weights.DEFAULT)

        original_conv = self.backbone.features[0][0]

        new_conv = nn.Conv2d(
            in_channels=1,
            out_channels=original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding
        )


        new_conv.weight.data = original_conv.weight.data.mean(dim=1, keepdim=True)
        if original_conv.bias is not None:
            new_conv.bias.data = original_conv.bias.data

        self.backbone.features[0][0] = new_conv

        in_features = self.backbone.head.in_features
        self.backbone.head = nn.Identity()

        self.projection_head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, out_channels)
        )

    def forward(self, x):
        """
        x: Spectrogram Tensor of shape [batch_size, 1, 92, num_frames]
        """

        H, W = x.shape[2], x.shape[3]
        pad_h = (32 - H % 32) % 32
        pad_w = (32 - W % 32) % 32

        x = F.pad(x, (0, pad_w, 0, pad_h))

        features = self.backbone(x)

        projected = self.projection_head(features)

        out = F.normalize(projected, p=2, dim=1)

        return out

In [ ]:
class SymmetricCrossModalMoCo(nn.Module):
    def __init__(self, graph_encoder, audio_encoder, dim=512, K=65536, m=0.999, T=0.07):
        super(SymmetricCrossModalMoCo, self).__init__()

        self.K = K    # num negatives
        self.m = m    # momentum
        self.T = T    # temp

        self.encoder_q_graph = graph_encoder
        self.encoder_q_audio = audio_encoder

        self.encoder_k_graph = copy.deepcopy(graph_encoder)
        self.encoder_k_audio = copy.deepcopy(audio_encoder)

        for param_q, param_k in zip(self.encoder_q_graph.parameters(), self.encoder_k_graph.parameters()):
            param_k.requires_grad = False
        for param_q, param_k in zip(self.encoder_q_audio.parameters(), self.encoder_k_audio.parameters()):
            param_k.requires_grad = False

        self.register_buffer("queue_graph", torch.randn(dim, K))
        self.register_buffer("queue_audio", torch.randn(dim, K))
        self.queue_graph = F.normalize(self.queue_graph, dim=0)
        self.queue_audio = F.normalize(self.queue_audio, dim=0)

        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _momentum_update_key_encoders(self):
        for param_q, param_k in zip(self.encoder_q_graph.parameters(), self.encoder_k_graph.parameters()):
            param_k.data = param_k.data * self.m + param_q.data * (1. - self.m)

        for param_q, param_k in zip(self.encoder_q_audio.parameters(), self.encoder_k_audio.parameters()):
            param_k.data = param_k.data * self.m + param_q.data * (1. - self.m)

    @torch.no_grad()
    def _dequeue_and_enqueue(self, keys_graph, keys_audio):
        batch_size = keys_graph.shape[0]
        ptr = int(self.queue_ptr)

        if ptr + batch_size > self.K:
            overflow = (ptr + batch_size) - self.K
            fit = batch_size - overflow

            self.queue_graph[:, ptr:self.K] = keys_graph.T[:, :fit]
            self.queue_audio[:, ptr:self.K] = keys_audio.T[:, :fit]

            self.queue_graph[:, 0:overflow] = keys_graph.T[:, fit:]
            self.queue_audio[:, 0:overflow] = keys_audio.T[:, fit:]
        else:
            self.queue_graph[:, ptr:ptr + batch_size] = keys_graph.T
            self.queue_audio[:, ptr:ptr + batch_size] = keys_audio.T

        self.queue_ptr[0] = (ptr + batch_size) % self.K

    def forward(self, graph_inputs, audio_inputs):
        self._momentum_update_key_encoders()

        q_graph = self.encoder_q_graph(**graph_inputs)
        q_audio = self.encoder_q_audio(audio_inputs)

        with torch.no_grad():
            k_graph = self.encoder_k_graph(**graph_inputs)
            k_audio = self.encoder_k_audio(audio_inputs)


        l_pos_G2A = torch.einsum('nc,nc->n', [q_graph, k_audio]).unsqueeze(-1)
        l_neg_G2A = torch.einsum('nc,ck->nk', [q_graph, self.queue_audio.clone().detach()])
        logits_G2A = torch.cat([l_pos_G2A, l_neg_G2A], dim=1) / self.T


        l_pos_A2G = torch.einsum('nc,nc->n', [q_audio, k_graph]).unsqueeze(-1)
        l_neg_A2G = torch.einsum('nc,ck->nk', [q_audio, self.queue_graph.clone().detach()])
        logits_A2G = torch.cat([l_pos_A2G, l_neg_A2G], dim=1) / self.T

        labels = torch.zeros(logits_G2A.shape[0], dtype=torch.long).to(q_graph.device)
        loss_G2A = F.cross_entropy(logits_G2A, labels)
        loss_A2G = F.cross_entropy(logits_A2G, labels)

        total_loss = loss_G2A + loss_A2G

        self._dequeue_and_enqueue(k_graph, k_audio)

        return total_loss

    def train(self, mode=True):
        super().train(mode)
        self.encoder_k_graph.eval()
        self.encoder_k_audio.eval()
        return self

In [ ]:
@torch.no_grad()
def evaluate_retrieval(moco_model, val_loader, device='cuda'):
    moco_model.eval()

    all_graph_embeddings = []
    all_audio_embeddings = []
    all_line_ids = []

    for batch_data in tqdm(val_loader, desc="Validation Forward Pass"):
        audio_inputs = batch_data['spectrograms'].to(device)
        graph_inputs = {
            'x_cont': batch_data['graph_x_cont'].to(device),
            'x_class': batch_data['graph_x_class'].to(device),
            'x_pitch': batch_data['graph_x_pitch'].to(device),
            'edge_index': batch_data['graph_edge_index'].to(device),
            'batch': batch_data['graph_batch_index'].to(device)
        }

        q_graph = moco_model.encoder_q_graph(**graph_inputs)
        q_audio = moco_model.encoder_q_audio(audio_inputs)

        all_graph_embeddings.append(q_graph.cpu())
        all_audio_embeddings.append(q_audio.cpu())

        all_line_ids.extend(batch_data['line_id'])

    graph_embeds = torch.cat(all_graph_embeddings, dim=0)
    audio_embeds = torch.cat(all_audio_embeddings, dim=0)
    num_samples = graph_embeds.shape[0]

    print(f"Smilarity matrix for {num_samples} snippets...")
    similarity_matrix = torch.matmul(audio_embeds, graph_embeds.T)

    encoder = LabelEncoder()
    line_labels = encoder.fit_transform(all_line_ids)
    labels_tensor = torch.tensor(line_labels)

    ground_truth_mask = (labels_tensor.unsqueeze(1) == labels_tensor.unsqueeze(0))

    def calculate_ranks(sim_matrix):
        sorted_indices = torch.argsort(sim_matrix, dim=1, descending=True)

        sorted_mask = torch.gather(ground_truth_mask, 1, sorted_indices)

        ranks = sorted_mask.float().argmax(dim=1) + 1
        ranks = ranks.float()

        r1 = (ranks <= 1).float().mean().item()
        r25 = (ranks <= 25).float().mean().item()
        mrr = (1.0 / ranks).mean().item()
        mr = ranks.median().item()

        return r1, r25, mrr, mr

    metrics = {}
    r1, r25, mrr, mr = calculate_ranks(similarity_matrix)
    metrics['A2S'] = {'R@1': r1, 'R@25': r25, 'MRR': mrr, 'MR': mr}

    r1, r25, mrr, mr = calculate_ranks(similarity_matrix.T)
    metrics['S2A'] = {'R@1': r1, 'R@25': r25, 'MRR': mrr, 'MR': mr}

    moco_model.train()
    return metrics

In [ ]:
def load_checkpoint(checkpoint_path, moco_model, optimizer=None, device='cuda'):
    """Loads model weights, optimizer state, and training metadata."""
    print(f"Loading checkpoint from: {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    moco_model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    epoch = checkpoint.get('epoch', 0)
    train_loss = checkpoint.get('loss', None)
    val_metrics = checkpoint.get('metrics', None)

    print(f"Successfully restored checkpoint from Epoch {epoch + 1}.")
    if train_loss is not None:
        print(f"  -> Recorded Training Loss: {train_loss:.4f}")
    if val_metrics is not None:
        print(f"  -> Recorded A2S MRR:       {val_metrics['A2S']['MRR']:.4f}")

    return moco_model, optimizer, epoch, train_loss, val_metrics

def train_phase_1_moco(
    train_loader,
    val_loader,
    num_epochs=20,
    learning_rate=5e-4,
    resume_checkpoint='/content/drive/MyDrive/MSMD_Checkpoints/phase1_moco_best_loss.pth',
    save_dir='/content/drive/MyDrive/MSMD_Checkpoints',
    device='cuda' if torch.cuda.is_available() else 'cpu'
):
    print(f"Starting Phase 1 Training on device: {device}")

    os.makedirs(save_dir, exist_ok=True)
    print(f"Checkpoints will be saved to: {save_dir}")

    vocab_size = 72
    graph_encoder = SheetMusicTeacherGAT(num_classes=vocab_size).to(device)
    audio_encoder = SpectrogramSwin(out_channels=512).to(device)

    moco_model = SymmetricCrossModalMoCo(
        graph_encoder=graph_encoder,
        audio_encoder=audio_encoder,
        dim=512,
        K=16384
    ).to(device)

    optimizer = optim.AdamW(moco_model.parameters(), lr=learning_rate, weight_decay=1e-4)
    warmup = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=1000)
    cosine = CosineAnnealingLR(optimizer, T_max=num_epochs - 1)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[1])
    start_epoch = 0
    best_loss = float('inf')

    if resume_checkpoint and os.path.exists(resume_checkpoint):
        print("Loading from checkpoint...")
        moco_model, optimizer, start_epoch, prev_loss, _ = load_checkpoint(resume_checkpoint, moco_model, optimizer, device)
        if prev_loss is not None:
            best_loss = prev_loss

        start_epoch += 1

    for epoch in range(start_epoch, num_epochs):
        moco_model.train()
        total_loss = 0.0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for batch_idx, batch_data in enumerate(progress_bar):

            audio_inputs = batch_data['spectrograms'].to(device, non_blocking=True)

            if moco_model.training:
                freq_mask = T_audio.FrequencyMasking(freq_mask_param=15)
                time_mask = T_audio.TimeMasking(time_mask_param=30)

                # unbind splits the batch into a list, we mask each, then stack them back up
                audio_inputs = torch.stack([time_mask(freq_mask(x)) for x in audio_inputs.unbind(0)])


            graph_inputs = {
                'x_cont': batch_data['graph_x_cont'].to(device, non_blocking=True),
                'x_class': batch_data['graph_x_class'].to(device, non_blocking=True),
                'x_pitch': batch_data['graph_x_pitch'].to(device, non_blocking=True),
                'edge_index': batch_data['graph_edge_index'].to(device, non_blocking=True),
                'batch': batch_data['graph_batch_index'].to(device, non_blocking=True)
            }
            if moco_model.training:
                jitter = (torch.rand_like(graph_inputs['x_cont'][:, :2]) - 0.5) * 0.02

                graph_inputs['x_cont'][:, :2] = torch.clamp(graph_inputs['x_cont'][:, :2] + jitter, 0.0, 1.0)


            optimizer.zero_grad()
            loss = moco_model(graph_inputs, audio_inputs)
            loss.backward()
            clip_grad_norm_(moco_model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()


            total_loss += loss.item()
            progress_bar.set_postfix({'Loss': f"{loss.item():.4f}"})
            if batch_idx % 100 == 0:
                print(f"  [Debug] ")

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} Completed | Average Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

        if avg_loss < best_loss:
            print(f"best loss: ({best_loss:.4f} -> {avg_loss:.4f}) ***")
            best_loss = avg_loss

            best_checkpoint_path = os.path.join(save_dir, "phase1_moco_best_loss.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': moco_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_loss,
            }, best_checkpoint_path)

        print("\n--- Validation Retrieval ---")
        metrics = evaluate_retrieval(moco_model, val_loader, device)

        print("Audio-to-Score:")
        print(f"  R@1: {metrics['A2S']['R@1']:.3f} | R@25: {metrics['A2S']['R@25']:.3f} | MRR: {metrics['A2S']['MRR']:.3f} | MR: {metrics['A2S']['MR']}")

        print("Score-to-Audio:")
        print(f"  R@1: {metrics['S2A']['R@1']:.3f} | R@25: {metrics['S2A']['R@25']:.3f} | MRR: {metrics['S2A']['MRR']:.3f} | MR: {metrics['S2A']['MR']}")
        print("------------------------------------\n")

        interval_checkpoint_path = os.path.join(save_dir, f"phase1_moco_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch,
            'model_state_dict': moco_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
            'metrics': metrics,
        }, interval_checkpoint_path)

    print("Phase 1 Training Complete.")
    return moco_model

In [ ]:
# train_phase_1_moco(train_loader=train_loader, val_loader=val_loader)

### Stage 2
Vision-Graph Pretraining

In [ ]:
@torch.no_grad()
def evaluate_retrieval_phase2(vision_student, audio_teacher, val_loader, device='cuda'):
    vision_student.eval()
    audio_teacher.eval()

    all_vision_embeddings = []
    all_audio_embeddings = []
    all_line_ids = []

    for batch_data in tqdm(val_loader, desc="Validation Forward Pass"):
        images = batch_data['images'].to(device)  # Matches dataset key
        audio_inputs = batch_data['spectrograms'].to(device)

        q_audio = audio_teacher(audio_inputs)
        q_vision = vision_student(images)

        all_vision_embeddings.append(q_vision.cpu())
        all_audio_embeddings.append(q_audio.cpu())
        all_line_ids.extend(batch_data['line_id'])

    vision_embeds = torch.cat(all_vision_embeddings, dim=0)
    audio_embeds = torch.cat(all_audio_embeddings, dim=0)

    unique_line_ids = []
    unique_vision_indices = []
    seen = set()
    for idx, lid in enumerate(all_line_ids):
        if lid not in seen:
            seen.add(lid)
            unique_line_ids.append(lid)
            unique_vision_indices.append(idx)

    # Keep only one unique vision embedding per sheet music line
    dedup_vision_embeds = vision_embeds[unique_vision_indices]

    similarity_matrix = torch.matmul(audio_embeds, dedup_vision_embeds.T)

    encoder = LabelEncoder()
    encoder.fit(unique_line_ids)

    query_labels = torch.tensor(encoder.transform(all_line_ids))      # Shape: [TotalAudioQueries]
    gallery_labels = torch.tensor(encoder.transform(unique_line_ids))  # Shape: [NumUniqueLines]

    ground_truth_mask = (query_labels.unsqueeze(1) == gallery_labels.unsqueeze(0))

    def calculate_ranks(sim_matrix, mask):
        sorted_indices = torch.argsort(sim_matrix, dim=1, descending=True)
        sorted_mask = torch.gather(mask, 1, sorted_indices)

        ranks = sorted_mask.float().argmax(dim=1) + 1
        ranks = ranks.float()

        r1 = (ranks <= 1).float().mean().item()
        r25 = (ranks <= 25).float().mean().item()
        mrr = (1.0 / ranks).mean().item()
        mr = ranks.median().item()
        return r1, r25, mrr, mr

    metrics = {}
    r1, r25, mrr, mr = calculate_ranks(similarity_matrix, ground_truth_mask)
    metrics['A2V'] = {'R@1': r1, 'R@25': r25, 'MRR': mrr, 'MR': mr}

    r1, r25, mrr, mr = calculate_ranks(similarity_matrix.T, ground_truth_mask.T)
    metrics['V2A'] = {'R@1': r1, 'R@25': r25, 'MRR': mrr, 'MR': mr}

    vision_student.train()
    return metrics

In [ ]:
def train_phase_2_distillation(
    train_loader,
    val_loader,
    graph_teacher,
    audio_teacher,
    num_epochs=30,
    learning_rate=3e-4,
    save_dir='/content/drive/MyDrive/MSMD_Checkpoints',
    resume_checkpoint='/content/drive/MyDrive/MSMD_Checkpoints/phase2_vision_student_best.pth',
    device='cuda' if torch.cuda.is_available() else 'cpu'
):
    os.makedirs(save_dir, exist_ok=True)

    vision_student = SheetMusicSwin(out_channels=512).to(device)

    graph_teacher.eval()
    audio_teacher.eval()
    for p in graph_teacher.parameters(): p.requires_grad = False
    for p in audio_teacher.parameters(): p.requires_grad = False

    optimizer = optim.AdamW(vision_student.parameters(), lr=learning_rate, weight_decay=1e-4)
    warmup = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=1000)
    cosine = CosineAnnealingLR(optimizer, T_max=num_epochs - 1)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[1])

    start_epoch = 0
    best_loss = float('inf')

    if resume_checkpoint and os.path.exists(resume_checkpoint):
        print("Loading from checkpoint...")
        vision_student, optimizer, start_epoch, prev_loss, _ = load_checkpoint(resume_checkpoint, vision_student, optimizer, device)
        if prev_loss is not None:
            best_loss = prev_loss
        start_epoch += 1

    vision_augmenter = T.Compose([
        T.RandomAdjustSharpness(sharpness_factor=2, p=0.5),
        T.ColorJitter(brightness=0.2, contrast=0.2),
        T.RandomAffine(degrees=1, translate=(0.02, 0.02))
    ])

    for epoch in range(start_epoch, num_epochs):
        vision_student.train()
        total_epoch_loss = 0.0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for batch_idx, batch_data in enumerate(progress_bar):
            images = batch_data['images'].to(device, non_blocking=True)

            graph_inputs = {
                'x_cont': batch_data['graph_x_cont'].to(device, non_blocking=True),
                'x_class': batch_data['graph_x_class'].to(device, non_blocking=True),
                'x_pitch': batch_data['graph_x_pitch'].to(device, non_blocking=True),
                'edge_index': batch_data['graph_edge_index'].to(device, non_blocking=True),
                'batch': batch_data['graph_batch_index'].to(device, non_blocking=True)
            }

            if vision_student.training:
                images = vision_augmenter(images)

            optimizer.zero_grad()

            with torch.no_grad():
                target_g_embed = graph_teacher(**graph_inputs)

            v_embed = vision_student(images)

            loss = 1.0 - F.cosine_similarity(v_embed, target_g_embed, dim=-1).mean()

            loss.backward()
            clip_grad_norm_(vision_student.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            total_epoch_loss += loss.item()

            progress_bar.set_postfix({'Loss': f"{loss.item():.4f}"})

            if batch_idx % 25 == 0:
                print(f"  [Debug] Current Loss: {loss.item():.4f}")

        avg_loss = total_epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1} | Avg Total Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

        if avg_loss < best_loss:
            print(f"*** New Best Loss: ({best_loss:.4f} -> {avg_loss:.4f}) ***")
            best_loss = avg_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': vision_student.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_loss,
            }, os.path.join(save_dir, "phase2_vision_student_best.pth"))

        print("\n--- Phase 2 Validation Retrieval ---")
        metrics = evaluate_retrieval_phase2(vision_student, audio_teacher, val_loader, device)

        print("Audio-to-Vision:")
        print(f"  R@1: {metrics['A2V']['R@1']:.3f} | R@25: {metrics['A2V']['R@25']:.3f} | MRR: {metrics['A2V']['MRR']:.3f} | MR: {metrics['A2V']['MR']}")
        print("Vision-to-Audio:")
        print(f"  R@1: {metrics['V2A']['R@1']:.3f} | R@25: {metrics['V2A']['R@25']:.3f} | MRR: {metrics['V2A']['MRR']:.3f} | MR: {metrics['V2A']['MR']}")
        print("------------------------------------\n")

    print("Phase 2 Training Complete.")
    return vision_student

In [ ]:
# device = 'cuda'
# vocab_size = 72
# checkpoint_path = '/content/drive/MyDrive/MSMD_Checkpoints/phase1_moco_best_loss.pth'

# graph_encoder = SheetMusicTeacherGAT(num_classes=vocab_size).to(device)
# audio_encoder = SpectrogramSwin(out_channels=512).to(device)
# moco_model_old = SymmetricCrossModalMoCo(graph_encoder=graph_encoder, audio_encoder=audio_encoder, dim=512, K=16384).to(device)
# moco_model_old, *rest  = load_checkpoint(checkpoint_path, moco_model_old, optimizer=None, device='cuda')

# graph_encoder = moco_model_old.encoder_q_graph
# audio_encoder = moco_model_old.encoder_q_audio

# train_phase_2_distillation(
#     train_loader=train_loader,
#     val_loader=val_loader,
#     graph_teacher=graph_encoder,
#     audio_teacher=audio_encoder,
#     num_epochs=30
# )

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

### Stage 3
Vision-Audio Fine-Tuning

In [ ]:
class VisionAudioMoCo(nn.Module):
    def __init__(self, vision_encoder, audio_encoder, dim=512, K=16384, m=0.999, T=0.07):
        super(VisionAudioMoCo, self).__init__()

        self.K = K
        self.m = m
        self.T = T

        self.encoder_q_vision = vision_encoder
        self.encoder_q_audio = audio_encoder

        self.encoder_k_vision = copy.deepcopy(vision_encoder)
        self.encoder_k_audio = copy.deepcopy(audio_encoder)

        for param_q, param_k in zip(self.encoder_q_vision.parameters(), self.encoder_k_vision.parameters()):
            param_k.requires_grad = False
        for param_q, param_k in zip(self.encoder_q_audio.parameters(), self.encoder_k_audio.parameters()):
            param_k.requires_grad = False

        self.register_buffer("queue_vision", torch.randn(dim, K))
        self.register_buffer("queue_audio", torch.randn(dim, K))
        self.queue_vision = F.normalize(self.queue_vision, dim=0)
        self.queue_audio = F.normalize(self.queue_audio, dim=0)

        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _momentum_update_key_encoders(self):
        for param_q, param_k in zip(self.encoder_q_vision.parameters(), self.encoder_k_vision.parameters()):
            param_k.data = param_k.data * self.m + param_q.data * (1. - self.m)
        for param_q, param_k in zip(self.encoder_q_audio.parameters(), self.encoder_k_audio.parameters()):
            param_k.data = param_k.data * self.m + param_q.data * (1. - self.m)

    @torch.no_grad()
    def _dequeue_and_enqueue(self, keys_vision, keys_audio):
        batch_size = keys_vision.shape[0]
        ptr = int(self.queue_ptr)

        self.queue_vision[:, ptr:ptr + batch_size] = keys_vision.T
        self.queue_audio[:, ptr:ptr + batch_size] = keys_audio.T

        self.queue_ptr[0] = (ptr + batch_size) % self.K

    def forward(self, images, audio_inputs):
        self._momentum_update_key_encoders()

        q_vision = self.encoder_q_vision(images)
        q_audio = self.encoder_q_audio(audio_inputs)

        with torch.no_grad():
            k_vision = self.encoder_k_vision(images)
            k_audio = self.encoder_k_audio(audio_inputs)

        # Vision -> Audio
        l_pos_V2A = torch.einsum('nc,nc->n', [q_vision, k_audio]).unsqueeze(-1)
        l_neg_V2A = torch.einsum('nc,ck->nk', [q_vision, self.queue_audio.clone().detach()])
        logits_V2A = torch.cat([l_pos_V2A, l_neg_V2A], dim=1) / self.T

        # Audio -> Vision
        l_pos_A2V = torch.einsum('nc,nc->n', [q_audio, k_vision]).unsqueeze(-1)
        l_neg_A2V = torch.einsum('nc,ck->nk', [q_audio, self.queue_vision.clone().detach()])
        logits_A2V = torch.cat([l_pos_A2V, l_neg_A2V], dim=1) / self.T

        labels = torch.zeros(logits_V2A.shape[0], dtype=torch.long).to(q_vision.device)
        loss_V2A = F.cross_entropy(logits_V2A, labels)
        loss_A2V = F.cross_entropy(logits_A2V, labels)

        self._dequeue_and_enqueue(k_vision, k_audio)

        return loss_V2A + loss_A2V

    def train(self, mode=True):
        super().train(mode)
        self.encoder_k_vision.eval()
        self.encoder_k_audio.eval()
        return self

In [ ]:
def train_phase_3(
    train_loader,
    val_loader,
    vision_student,
    audio_teacher,
    num_epochs=30,
    learning_rate=5e-4,
    save_dir='/content/drive/MyDrive/MSMD_Checkpoints',
    resume_checkpoint='/content/drive/MyDrive/MSMD_Checkpoints/phase3_moco_best.pth',
    device='cuda' if torch.cuda.is_available() else 'cpu'
):
    print(f"Starting Phase 3 Joint MoCo Fine-Tuning on: {device}")
    os.makedirs(save_dir, exist_ok=True)

    moco_model = aVisionAudioMoCo(
        vision_encoder=vision_student,
        audio_encoder=audio_teacher,
        dim=512,
        K=16384
    ).to(device)

    optimizer = optim.AdamW(moco_model.parameters(), lr=learning_rate, weight_decay=1e-4)
    warmup = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=1000)
    cosine = CosineAnnealingLR(optimizer, T_max=num_epochs - 1)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[1])

    start_epoch = 0
    best_loss = float('inf')

    if resume_checkpoint and os.path.exists(resume_checkpoint):
        print("Loading from checkpoint...")
        moco_model, optimizer, start_epoch, prev_loss, _ = load_checkpoint(resume_checkpoint, moco_model, optimizer, device)
        if prev_loss is not None:
            best_loss = prev_loss
        start_epoch += 1
    vision_student = torch.compile(vision_student)

    vision_augmenter = T.Compose([
        T.RandomAdjustSharpness(sharpness_factor=2, p=0.5),
        T.ColorJitter(brightness=0.2, contrast=0.2),
        T.RandomAffine(degrees=1, translate=(0.02, 0.02))
    ])

    for epoch in range(start_epoch, num_epochs):
        moco_model.train()
        total_epoch_loss = 0.0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for batch_idx, batch_data in enumerate(progress_bar):
            images = batch_data['images'].to(device, non_blocking=True)
            audio_inputs = batch_data['spectrograms'].to(device, non_blocking=True)

            if moco_model.training:
                images = vision_augmenter(images)

                freq_mask = T_audio.FrequencyMasking(freq_mask_param=15)
                time_mask = T_audio.TimeMasking(time_mask_param=30)
                audio_inputs = torch.stack([time_mask(freq_mask(x)) for x in audio_inputs.unbind(0)])

            optimizer.zero_grad()

            loss = moco_model(images, audio_inputs)

            loss.backward()
            clip_grad_norm_(moco_model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            total_epoch_loss += loss.item()

            progress_bar.set_postfix({'Loss': f"{loss.item():.4f}"})
            if batch_idx % 50 == 0:
                print("[Debug]")

        avg_loss = total_epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1} | Avg Total Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

        if avg_loss < best_loss:
            print(f"*** New Best Loss: ({best_loss:.4f} -> {avg_loss:.4f}) ***")
            best_loss = avg_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': moco_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_loss,
            }, os.path.join(save_dir, "phase3_moco_best.pth"))

        print("\n--- Phase 3 Validation Retrieval ---")
        # Extract the active queries out of the MoCo wrapper for evaluation
        metrics = evaluate_retrieval_phase2(
            moco_model.encoder_q_vision,
            moco_model.encoder_q_audio,
            val_loader,
            device
        )

        print("Audio-to-Vision:")
        print(f"  R@1: {metrics['A2V']['R@1']:.3f} | R@25: {metrics['A2V']['R@25']:.3f} | MRR: {metrics['A2V']['MRR']:.3f} | MR: {metrics['A2V']['MR']}")
        print("Vision-to-Audio:")
        print(f"  R@1: {metrics['V2A']['R@1']:.3f} | R@25: {metrics['V2A']['R@25']:.3f} | MRR: {metrics['V2A']['MRR']:.3f} | MR: {metrics['V2A']['MR']}")
        print("------------------------------------\n")

    print("Phase 3 Training Complete.")
    return moco_model

In [ ]:
# device = 'cuda'
vocab_size = 72
checkpoint_path = '/content/drive/MyDrive/MSMD_Checkpoints/phase1_moco_best_loss.pth'
graph_encoder = SheetMusicTeacherGAT(num_classes=vocab_size).to(device)
audio_encoder = SpectrogramSwin(out_channels=512).to(device)
moco_model_old = SymmetricCrossModalMoCo(graph_encoder=graph_encoder, audio_encoder=audio_encoder, dim=512, K=16384).to(device)
moco_model_old, *rest  = load_checkpoint(checkpoint_path, moco_model_old, optimizer=None, device='cuda')
audio_encoder = moco_model_old.encoder_q_audio

checkpoint_path2 = '/content/drive/MyDrive/MSMD_Checkpoints/phase2_vision_student_best.pth'
vision_student = SheetMusicSwin(out_channels=512).to(device)
vision_student, *_ = load_checkpoint(checkpoint_path2, vision_student, optimizer=None, device='cuda')

train_phase_3(
    train_loader=train_loader,
    val_loader=val_loader,
    vision_student = vision_student,
    audio_teacher=audio_encoder,
    num_epochs=30,
    learning_rate=1e-4)





In [ ]:
device = 'cuda'
vision_student = SheetMusicSwin(out_channels=512).to(device)
audio_encoder = SpectrogramSwin(out_channels=512).to(device)

moco_model = VisionAudioMoCo(
        vision_encoder=vision_student,
        audio_encoder=audio_encoder,
        dim=512,
        K=16384).to(device)
resume_checkpoint='/content/drive/MyDrive/MSMD_Checkpoints/phase3_moco_best.pth'
save_dir = '/content/drive/MyDrive/MSMD_Checkpoints/'
moco_model, * _ = load_checkpoint(resume_checkpoint, moco_model)
vision_student = moco_model.encoder_q_vision
audio_encoder = moco_model.encoder_q_audio
metrics = evaluate_retrieval_phase2(
    moco_model.encoder_q_vision,
    moco_model.encoder_q_audio,
    test_loader,
    device
)

print("Audio-to-Vision:")
print(f"  R@1: {metrics['A2V']['R@1']:.3f} | R@25: {metrics['A2V']['R@25']:.3f} | MRR: {metrics['A2V']['MRR']:.3f} | MR: {metrics['A2V']['MR']}")
print("Vision-to-Audio:")
print(f"  R@1: {metrics['V2A']['R@1']:.3f} | R@25: {metrics['V2A']['R@25']:.3f} | MRR: {metrics['V2A']['MRR']:.3f} | MR: {metrics['V2A']['MR']}")
print("------------------------------------\n")



###Get model state dicts


In [ ]:
device='cuda'
vision_student = SheetMusicSwin(out_channels=512).to(device)
audio_encoder = SpectrogramSwin(out_channels=512).to(device)

moco_model = VisionAudioMoCo(
        vision_encoder=vision_student,
        audio_encoder=audio_encoder,
        dim=512,
        K=16384).to(device)
resume_checkpoint='/content/drive/MyDrive/MSMD_Checkpoints/phase3_moco_best.pth'
save_dir = '/content/drive/MyDrive/MSMD_Checkpoints/'
moco_model, * _ = load_checkpoint(resume_checkpoint, moco_model)
vision_student = moco_model.encoder_q_vision
audio_encoder = moco_model.encoder_q_audio

torch.save({'vision_state_dict': vision_student.state_dict(),
            'audio_state_dict': audio_encoder.state_dict()}, os.path.join(save_dir, "vision_audio_model.pth"))